In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas

In [ ]:
def rotatedGauss(x, y, x0, y0, s_x, s_y, theta):
    a = np.cos(theta)**2/(2*s_x**2) + np.sin(theta)**2/(2*s_y**2)
    b = -np.sin(theta)*np.cos(theta)/(2*s_x**2) + np.sin(theta)*np.cos(theta)/(2*s_y**2)
    c = np.sin(theta)**2/(2*s_x**2) + np.cos(theta)**2/(2*s_y**2)
    
    return np.exp(-(a*(x - x0)**2 + 2*b*(x-x0)*(y-y0) + c*(y-y0)**2))

In [ ]:
x, y = np.arange(-10,10), np.arange(-10,10)

X, Y = np.meshgrid(x, y)

plt.imshow(rotatedGauss(X, Y, 0, 0, 3, 1, np.pi/10))

In [ ]:
def centroid(frame):
    i,j = np.arange(frame.shape[0]), np.arange(frame.shape[1])
    I,J = np.meshgrid(i, j)
    frame_sum = np.sum(frame)
    return np.sum(frame*I)/frame_sum, np.sum(frame*J)/frame_sum

In [ ]:

def generateRandomBlob(x, y, ng=None):
    out = np.zeros_like(x)
    X0, Y0 = np.random.random(2)*3 - 1.5
    NG = ng
    if ng is None:
        NG = np.random.randint(1,5)
    
    theta = np.random.uniform(-0.05,0.05)
    for _ in range(NG):

        s_x = np.random.uniform(1.0,1.4)
        s_y = np.random.uniform(1.4,2.0)
        
        core = rotatedGauss(x, y, X0, Y0, s_x, s_y, theta)
        halo = rotatedGauss(x, y, X0, Y0, s_x * 2.0, s_y * 2.0, theta)

        a_core = np.random.uniform(0.8, 1.2)
        a_halo = np.random.uniform(0.05, 0.15)

        out = out + (a_core * core) + a_halo * halo

    return out

In [ ]:
import numpy as np

N_BLOBS = 100_000

blobs = np.zeros((N_BLOBS, 19200))
cent = np.zeros((N_BLOBS, 16, 2))
integ = np.zeros((N_BLOBS,16))
binary = np.zeros((N_BLOBS,16))

for run in range(N_BLOBS):
    num = np.random.randint(1, 17)
    k_start = np.random.randint(2, 8)
    
    blobby = np.zeros((30, 640))

    info = np.zeros((16, 2))
    info2 = np.zeros(16)
    info3 = np.zeros(16)


    x_centers = np.linspace(30, 610, num).astype(int)

    for i in range(num):
        ng = np.random.randint(1, 5)
        blob = generateRandomBlob(X, Y, ng=ng) / ng
        h, w = blob.shape  
        
    
        j_start = x_centers[i] - (w // 2)
        

        j_start = max(0, min(j_start, 640 - w))
       
        current_k = min(k_start, 30 - h)

        blobby[current_k : current_k + h, j_start : j_start + w] += blob


        local_c = centroid(blob)
        info[i, 0] = local_c[0] + j_start
        info[i, 1] = local_c[1] + current_k
        info2[i] = np.sum(blob)
        info3[i] = 1


    blobs[run] = blobby.flatten()
    cent[run] = info
    integ[run] = info2
    binary[run] = info3


In [ ]:
import joblib
saveDat = {'blobs': blobs,
           'center': cent,
           'intensity': integ,
           'classify': binary}
with open("New_MultiBlob_TRA_DAT.joblib", 'wb') as f:
    joblib.dump(saveDat, f)